#Create Embedding Vectors for GoEmotions

**Author:** Thor Lemke

The goal of this file is to create and save the vector embeddings of all text in the simplified GoEmotions dataset.  These embeddings are pooled for consistent size, resulting in all vectors being 768 elements long (from BERT architecture, not a value we selected).  The embeddings are stored in separate files for each split to be used further along in the ML pipeline.  A primary reason for saving the values in a csv is time, as running the embedding for the entire training data was a process that took well over an hour on CoLab.

In [ ]:
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Import the three splits of the simplified dataset for GoEmotions
ds_train = load_dataset("google-research-datasets/go_emotions", "simplified", split="train")
ds_validate = load_dataset("google-research-datasets/go_emotions", "simplified", split="validation")
ds_test = load_dataset("google-research-datasets/go_emotions", "simplified", split="test")

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

simplified/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

simplified/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  347kB            

simplified/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [ ]:
from transformers import BertTokenizer, BertModel

In [ ]:
# Load the base BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# For all training points, find the embedding vectore (pooled for consistent
# size) of the text, and store the results for later use

NUM_VALUES_PER_EMBEDDING = 768

features_matrix = np.zeros((ds_train.num_rows, NUM_VALUES_PER_EMBEDDING))

for i in range(ds_train.num_rows):
  text = ds_train[i]["text"]
  encoded_input = tokenizer(text, return_tensors='pt')
  output = model(**encoded_input)
  features_matrix[i, :] = output.pooler_output.detach().numpy()

np.savetxt("features_matrix.csv", features_matrix, delimiter=",")

In [ ]:
# Now find the embedding vectors for all validation data points

NUM_VALUES_PER_EMBEDDING = 768

features_matrix_validate = np.zeros((ds_validate.num_rows, NUM_VALUES_PER_EMBEDDING))

for i in range(ds_validate.num_rows):
  text = ds_validate[i]["text"]
  encoded_input = tokenizer(text, return_tensors='pt')
  output = model(**encoded_input)
  features_matrix_validate[i, :] = output.pooler_output.detach().numpy()

np.savetxt("features_matrix_validate.csv", features_matrix_validate, delimiter=",")


In [ ]:
# Finally, find the embedding vectors for all test data points

NUM_VALUES_PER_EMBEDDING = 768

features_matrix_test = np.zeros((ds_test.num_rows, NUM_VALUES_PER_EMBEDDING))

for i in range(ds_test.num_rows):
  text = ds_test[i]["text"]
  encoded_input = tokenizer(text, return_tensors='pt')
  output = model(**encoded_input)
  features_matrix_test[i, :] = output.pooler_output.detach().numpy()

np.savetxt("features_matrix_test.csv", features_matrix_test, delimiter=",")